file largely AI generated

# Do our hand-authored stubs cover the matplotlib functions people actually read about?

Joins two artifacts that live in this folder:

1. `stubs-e328827.ai-then-human-annotated.diff` — our stub fork diffed against upstream
   `microsoft/python-type-stubs@e328827`, with one `### ANN` line per changed function
   classifying it `MEANINGFUL` (real hand-authoring that changes what the GUI renders) or
   `COSMETIC`. See `README.md` and `ANNOTATION_GUIDE.md`.
2. `top matplotlib pages as of 2026-07-24 ….csv` — the 400 most-visited pages on
   matplotlib.org over the trailing 28 days, from views.scientific-python.org.

Goal: fill in **X** and **Y** in

> We hand-authored annotations for **X** functions, which, though a vanishingly small portion of
> matplotlib's entire API, accounts for **Y**% of the top 100 most-visited matplotlib function
> documentation pages.

**Answer: X = 47, Y = 30.** A page only counts if a user who writes the call that page documents
actually gets a hand-annotated panel in Plottery, which makes the match exact on the calling
surface — annotating `Axes.plot` does nothing for someone who typed `plt.plot`, verified in the live
UI in §6 (4 property rows vs 35). §8 measures what that rule costs us: 29 further pages are waiting
only on an un-annotated `pyplot` wrapper.

In [1]:
import importlib
import inspect
import re
import types
from pathlib import Path

import pandas as pd

HERE = Path.cwd()
if HERE.name != "determine_annotation_coverage":
    HERE = (HERE / "Analysis/determine_annotation_coverage").resolve()

DIFF = HERE / "stubs-e328827.ai-then-human-annotated.diff"
CSV = HERE / "top matplotlib pages as of 2026-07-24 for analyzing our type stub coverage.csv"
assert DIFF.exists(), DIFF
assert CSV.exists(), CSV

pd.set_option("display.max_rows", 200, "display.width", 200, "display.max_colwidth", 80)
print(HERE)

/Users/brian/Documents/ucsd/bimodal_plotting/plottery/Analysis/determine_annotation_coverage


## 1. Extract the `### ANN` annotations from the diff

In [2]:
ANN_RE = re.compile(
    r"^### ANN \s*\|\s*(?P<name>[^|]+?)\s*\|\s*(?P<verdict>MEANINGFUL|COSMETIC)\s*"
    r"\|\s*(?P<tags>[^|]*?)\s*\|\s*(?P<note>.*)$"
)

records, unparsed = [], []
for line in DIFF.read_text().splitlines():
    if not line.startswith("### ANN"):
        continue
    m = ANN_RE.match(line)
    if m is None:
        unparsed.append(line)
        continue
    records.append(
        {
            "name": m["name"],
            "verdict": m["verdict"],
            "tags": [t for t in m["tags"].split(",") if t],
            "note": m["note"],
        }
    )

ann = pd.DataFrame(records)
print(f"{len(ann)} annotations parsed, {len(unparsed)} unparsed")
for line in unparsed:
    print("  UNPARSED:", line)
print(ann["verdict"].value_counts().to_string())

115 annotations parsed, 0 unparsed
verdict
COSMETIC      68
MEANINGFUL    47


## 2. The list of MEANINGFUL (hand-annotated) function names

In [3]:
dupes = ann["name"].value_counts()
print("names annotated more than once:", dict(dupes[dupes > 1]) or "none")

hand = sorted(set(ann.loc[ann["verdict"] == "MEANINGFUL", "name"]))
cosmetic_only = sorted(set(ann.loc[ann["verdict"] == "COSMETIC", "name"]) - set(hand))

print(f"\n{len(hand)} distinct hand-annotated (MEANINGFUL) functions")
print(f"{len(cosmetic_only)} distinct functions changed cosmetically only")

module_of = lambda n: ".".join(n.split(".")[:-1])
print("\nby owner:")
print(pd.Series([module_of(n) for n in hand]).value_counts().to_string())
print()
for n in hand:
    print("  ", n)

names annotated more than once: {'matplotlib.axes.Axes.hist': 2}

47 distinct hand-annotated (MEANINGFUL) functions
67 distinct functions changed cosmetically only

by owner:
matplotlib.axes.Axes        39
matplotlib.figure.Figure     4
matplotlib.pyplot            4

   matplotlib.axes.Axes.annotate
   matplotlib.axes.Axes.axhline
   matplotlib.axes.Axes.axhspan
   matplotlib.axes.Axes.axis
   matplotlib.axes.Axes.axline
   matplotlib.axes.Axes.axvline
   matplotlib.axes.Axes.axvspan
   matplotlib.axes.Axes.bar
   matplotlib.axes.Axes.bar_label
   matplotlib.axes.Axes.barh
   matplotlib.axes.Axes.boxplot
   matplotlib.axes.Axes.eventplot
   matplotlib.axes.Axes.grid
   matplotlib.axes.Axes.hist
   matplotlib.axes.Axes.hlines
   matplotlib.axes.Axes.indicate_inset
   matplotlib.axes.Axes.inset_axes
   matplotlib.axes.Axes.legend
   matplotlib.axes.Axes.loglog
   matplotlib.axes.Axes.plot
   matplotlib.axes.Axes.plot_date
   matplotlib.axes.Axes.scatter
   matplotlib.axes.Axes.secondary

## 3. Load the page-traffic CSV

In [4]:
def parse_count(s: str) -> float:
    s = str(s).strip().replace(",", "")
    mult = {"k": 1e3, "m": 1e6}.get(s[-1:].lower(), 1)
    return float(s[:-1] if mult != 1 else s) * mult


pages = pd.read_csv(CSV, skiprows=2)  # first two lines are provenance comments
pages.columns = [c.replace("↓", "").strip() for c in pages.columns]
pages["visitors"] = pages["Visitors"].map(parse_count)
pages["site_rank"] = range(1, len(pages) + 1)

assert pages["visitors"].is_monotonic_decreasing, "CSV is not sorted by popularity"
print(f"{len(pages)} pages, {pages['visitors'].sum():,.0f} visitors total")
pages.head()

400 pages, 1,054,026 visitors total


,Page url,Visitors,Pageviews,Bounce rate,Time on page,Scroll depth,visitors,site_rank
0,https://matplotlib.org/stable/users/explain/colors/colormaps.html,63.5k,75.3k,81%,1m 10s,44%,63500.0,1
1,https://matplotlib.org/stable/gallery/color/named_colors.html,44k,52.2k,77%,47s,78%,44000.0,2
2,https://matplotlib.org/,39.8k,59.5k,42%,18s,53%,39800.0,3
3,https://matplotlib.org/stable/search.html,22.6k,44.5k,61%,18s,33%,22600.0,4
4,https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html,20.9k,26.4k,59%,47s,42%,20900.0,5


## 4. Which URLs are per-object API reference pages?

`/api/_as_gen/<dotted.name>.html` is Sphinx's `autosummary` output directory, so it is exactly the
set of pages that document one object. Double-checking the assumption below: every other page under
`/api/` is a module-level summary page (`axes_api.html`, `pyplot_summary.html`, …), and every page
outside `/api/` is a tutorial, gallery example, or release note. So `_as_gen` is necessary *and*
sufficient.

But `_as_gen` also generates **class** pages (`matplotlib.lines.Line2D.html`) and one **module**
page (`matplotlib.pyplot.html`), which are not function documentation. We separate those by
importing the object out of the installed matplotlib and asking what it is, falling back to the
capitalisation convention for names that no longer resolve.

In [5]:
others = pages.loc[
    pages["Page url"].str.contains("/api/") & ~pages["Page url"].str.contains("/_as_gen/"),
    "Page url",
]
print(f"{len(others)} non-_as_gen /api/ pages — confirming none of them documents a single object:")
for u in others:
    print("  ", u.split("/api/")[-1])

AS_GEN_RE = re.compile(r"/api/_as_gen/(?P<obj>[A-Za-z0-9_.]+?)\.html")
pages["obj"] = pages["Page url"].str.extract(AS_GEN_RE)
pages["version"] = pages["Page url"].str.extract(r"matplotlib\.org/([^/]+)/")

as_gen = pages[pages["obj"].notna()].copy()
print(f"\n{len(as_gen)} of {len(pages)} pages are _as_gen object pages")
print("versions seen:", dict(as_gen["version"].value_counts()))

31 non-_as_gen /api/ pages — confirming none of them documents a single object:
   markers_api.html
   index.html
   pyplot_summary.html
   text_api.html
   axes_api.html
   matplotlib_configuration_api.html
   figure_api.html
   legend_api.html
   colors_api.html
   ticker_api.html
   collections_api.html
   pyplot_api.html
   colorbar_api.html
   animation_api.html
   cm_api.html
   index
   dates_api.html
   transformations.html
   axis_api.html
   artist_api.html
   scale_api.html
   colorizer_api.html
   font_manager_api.html
   style_api.html
   patches_api.html
   image_api.html
   spines_api.html
   backend_bases_api.html
   contour_api.html
   toolkits/mplot3d/view_angles.html
   widgets_api.html

123 of 400 pages are _as_gen object pages
versions seen: {'stable': 120, '3.5.3': 1, '2.1.2': 1, '2.1.1': 1}


In [6]:
def resolve(dotted: str):
    parts = dotted.split(".")
    for i in range(len(parts), 0, -1):
        try:
            obj = importlib.import_module(".".join(parts[:i]))
        except ImportError:
            continue
        for p in parts[i:]:
            try:
                obj = getattr(obj, p)
            except AttributeError:
                return None
        return obj
    return None


def kind(dotted: str) -> str:
    obj = resolve(dotted)
    if obj is None:  # removed from current matplotlib; fall back to the naming convention
        return "class?" if dotted.split(".")[-1][:1].isupper() else "function?"
    if isinstance(obj, types.ModuleType):
        return "module"
    if inspect.isclass(obj):
        return "class"
    return "function" if callable(obj) else "other"


as_gen["kind"] = as_gen["obj"].map(kind)
print(as_gen["kind"].value_counts().to_string())
print("\nnot counted as function documentation:")
print(as_gen.loc[~as_gen["kind"].str.startswith("function"), ["site_rank", "obj", "kind"]].to_string(index=False))

kind
function    106
class        16
module        1

not counted as function documentation:
 site_rank                                       obj   kind
        39                   matplotlib.lines.Line2D  class
        73                matplotlib.colors.Colormap  class
        77                      matplotlib.axes.Axes  class
        81                  matplotlib.figure.Figure  class
        87                         matplotlib.pyplot module
        96              matplotlib.patches.Rectangle  class
       117                  matplotlib.patches.Patch  class
       129        matplotlib.animation.FuncAnimation  class
       130              matplotlib.gridspec.GridSpec  class
       162            matplotlib.markers.MarkerStyle  class
       193                 matplotlib.patches.Circle  class
       231               matplotlib.colors.Normalize  class
       280          matplotlib.colors.ListedColormap  class
       310 matplotlib.colors.LinearSegmentedColormap  class
       

## 5. The top 100 function documentation pages

One wrinkle: the same object can appear several times under different doc versions
(`/stable/`, `/2.1.1/`, …). Those are the same function's page, so we merge them, summing visitors
and keeping the best rank, *before* cutting at 100.

In [7]:
func_pages = as_gen[as_gen["kind"].str.startswith("function")]
merged = (
    func_pages.groupby("obj", as_index=False)
    .agg(visitors=("visitors", "sum"), site_rank=("site_rank", "min"), n_urls=("obj", "size"))
    .sort_values(["visitors", "site_rank"], ascending=[False, True])
    .reset_index(drop=True)
)
merged["rank"] = merged.index + 1

print("objects whose traffic was spread over several versioned URLs:")
print(merged[merged["n_urls"] > 1][["rank", "obj", "n_urls", "visitors"]].to_string(index=False))

top100 = merged.head(100).copy()
print(f"\n{len(merged)} distinct functions have a doc page in the top-400; taking the first 100.")
print(f"cutoff: rank 100 is {top100.iloc[-1]['obj']} at {top100.iloc[-1]['visitors']:,.0f} visitors")

objects whose traffic was spread over several versioned URLs:
 rank                    obj  n_urls  visitors
    1 matplotlib.pyplot.plot       3   22688.0

104 distinct functions have a doc page in the top-400; taking the first 100.
cutoff: rank 100 is matplotlib.axes.Axes.bar_label at 644 visitors


## 6. The matching rule: does the user actually get a GUI?

A page counts only if someone who writes the call that page documents gets a hand-annotated panel in
Plottery. That makes the match **exact on the fully-qualified name**, because the calling surface
decides which stub is read:

- `snp.py` runs stock mypy over the notebook with our stubs on `mypy_path`
  (`options.mypy_path = [".../python-type-stubs-main/stubs"]`), so `plt.plot(...)` resolves to
  `pyplot.pyi`'s `plot` and `ax.plot(...)` resolves to `axes/_axes.pyi`'s `Axes.plot`. They are
  different stubs.
- There is no `pyplot` → `Axes` remapping anywhere in `snp.py` or the frontend.
- There is no runtime fallback: `snp.py` only ever pulls a `__doc__` off the live object
  (`try_to_add_docstring_to_call`), never `inspect.signature`. If the stub says `*args`, the
  properties panel has nothing to show.

So annotating `Axes.plot` does **not** give `plt.plot` a good GUI, and §8 confirms our `pyplot.pyi`
still has `def plot(*args, scalex=..., scaley=..., data=...)`.

Confirmed in the live UI. Same figure, same data, two spellings of the same call:

| `plt.plot([1,2,3],[2,1,3])` | `ax.plot([1,2,3],[2,1,3])` |
|---|---|
| ![plt.plot panel](evidence_plt_plot_panel.png) | ![ax.plot panel](evidence_ax_plot_panel.png) |

`plt.plot` yields **4 rows** — `args`, `scalex`, `scaley`, `data` — and worse, it binds the y-data to
`scalex`, so the panel is not just thin but wrong. `ax.plot` yields **35 rows**: `x`, `y`, `fmt`,
`label`, `color`, `linewidth`, `linestyle`, `marker`, `markersize`, … , i.e. the `Line2DProps`
TypedDict expanded, with sliders and real defaults. That is the entire difference between a page
counting and not counting.

The looser "same function, either spelling" count is still computed below, but only to size the gap —
it is not the number to publish.

In [8]:
WRAPPER_PREFIXES = ("matplotlib.pyplot.", "matplotlib.axes.Axes.", "matplotlib.figure.Figure.")


def normalize(name: str) -> str:
    for p in WRAPPER_PREFIXES:
        if name.startswith(p) and "." not in name[len(p) :]:
            return name[len(p) :]
    return name


hand_exact = set(hand)
hand_norm = {normalize(n) for n in hand}

top100["hand_exact"] = top100["obj"].isin(hand_exact)
top100["hand_norm"] = top100["obj"].map(normalize).isin(hand_norm)
top100["matched_via"] = [
    (
        obj
        if exact
        else ", ".join(n for n in hand if normalize(n) == normalize(obj)) if norm else ""
    )
    for obj, exact, norm in zip(top100["obj"], top100["hand_exact"], top100["hand_norm"])
]

n_exact, n_norm = int(top100["hand_exact"].sum()), int(top100["hand_norm"].sum())
print(f"top-100 function pages whose own stub we hand-annotated: {n_exact}")
print(f"  (pages whose function we annotated under *some* spelling: {n_norm} — see section 8)")
top100[["rank", "obj", "visitors", "hand_exact", "hand_norm", "matched_via"]]

top-100 function pages whose own stub we hand-annotated: 30
  (pages whose function we annotated under *some* spelling: 51 — see section 8)


,rank,obj,visitors,hand_exact,hand_norm,matched_via
0,1,matplotlib.pyplot.plot,22688.0,False,True,matplotlib.axes.Axes.plot
1,2,matplotlib.pyplot.legend,17100.0,False,True,matplotlib.axes.Axes.legend
2,3,matplotlib.pyplot.scatter,16500.0,False,True,matplotlib.axes.Axes.scatter
3,4,matplotlib.pyplot.subplots,14000.0,True,True,matplotlib.pyplot.subplots
4,5,matplotlib.pyplot.hist,13900.0,False,True,matplotlib.axes.Axes.hist
5,6,matplotlib.pyplot.bar,8800.0,False,True,matplotlib.axes.Axes.bar
6,7,matplotlib.pyplot.savefig,8800.0,False,False,
7,8,matplotlib.pyplot.imshow,7700.0,False,False,
8,9,matplotlib.pyplot.colorbar,7600.0,True,True,matplotlib.pyplot.colorbar
9,10,matplotlib.pyplot.figure,7400.0,True,True,matplotlib.pyplot.figure


## 7. The three lists

Using the strict rule: a page is covered when the call it documents resolves to a stub we
hand-annotated. Where a page misses only because its `pyplot` wrapper went un-annotated, list B says
so inline.

In [9]:
MATCH = "hand_exact"  # the call the page documents must itself be hand-annotated

in_top100_annotated = top100[top100[MATCH]]
in_top100_missing = top100[~top100[MATCH]]

covered = set(top100.loc[top100[MATCH], "obj"])
annotated_outside = [n for n in hand if n not in covered]


def show(title, rows):
    print(f"\n{'=' * 78}\n{title}  ({len(rows)})\n{'=' * 78}")
    for r in rows:
        print(r)


show(
    "A. functions with hand annotations, in the top 100",
    [f"{r.rank:>4}  {r.obj:<50} {r.visitors:>9,.0f}" for r in in_top100_annotated.itertuples()],
)
show(
    "B. functions WITHOUT hand annotations, in the top 100",
    [
        f"{r.rank:>4}  {r.obj:<50} {r.visitors:>9,.0f}"
        + (f"   <- we annotated {r.matched_via}, but not this wrapper" if r.hand_norm else "")
        for r in in_top100_missing.itertuples()
    ],
)
merged["norm"] = merged["obj"].map(normalize)


def where_documented(name: str) -> str:
    own = merged[merged["obj"] == name]
    if not own.empty:
        r = own.iloc[0]
        return f"its own page is rank {r['rank']} ({r['visitors']:,.0f} visitors) — below the cutoff"
    twin = merged[merged["norm"] == normalize(name)]
    if twin.empty:
        return "no doc page in the top 400, under any spelling"
    r = twin.iloc[0]
    return f"only its twin {r['obj']} is documented (rank {r['rank']})"


show(
    "C. functions with hand annotations, NOT in the top 100",
    [f"      {n:<40} {where_documented(n)}" for n in annotated_outside],
)


A. functions with hand annotations, in the top 100  (30)
   4  matplotlib.pyplot.subplots                            14,000
   9  matplotlib.pyplot.colorbar                             7,600
  10  matplotlib.pyplot.figure                               7,400
  13  matplotlib.axes.Axes.plot                              5,500
  17  matplotlib.axes.Axes.legend                            3,900
  20  matplotlib.axes.Axes.scatter                           3,400
  22  matplotlib.axes.Axes.text                              3,400
  26  matplotlib.axes.Axes.set_xticks                        2,800
  29  matplotlib.axes.Axes.tick_params                       2,500
  31  matplotlib.axes.Axes.set_xlim                          2,300
  32  matplotlib.figure.Figure.add_subplot                   2,200
  34  matplotlib.axes.Axes.set_title                         2,100
  36  matplotlib.axes.Axes.annotate                          2,000
  38  matplotlib.axes.Axes.bar                               1,900
  40

## 8. Sizing the gap: the un-annotated `pyplot` wrappers

21 pages in list B document a function we *did* hand-annotate, on the `Axes`/`Figure` side, and lost
only because the `pyplot` wrapper never got the same treatment. Below is our fork's actual
`pyplot.pyi` signature for each, which is what the user's `plt.…` call renders from.

This both justifies excluding them and measures what fixing them is worth.

In [10]:
FORK = HERE.parents[1] / "src/python-type-stubs-main/stubs/matplotlib"
PYPLOT_PYI = (FORK / "pyplot.pyi").read_text()


def stub_signature(text: str, name: str, indent: str = "") -> str | None:
    m = re.search(rf"^{indent}def {re.escape(name)}\(", text, re.M)
    if m is None:
        return None
    depth, i = 0, m.end() - 1
    while i < len(text):
        depth += (text[i] == "(") - (text[i] == ")")
        if depth == 0:
            return text[m.start() : i + 1]
        i += 1
    return None


def stub_shape(sig: str | None) -> str:
    if sig is None:
        return "absent"
    body = sig[sig.index("(") + 1 :]
    named = [p for p in re.split(r",(?![^\[\]]*\])", body) if p.strip().strip("()") not in ("", "*")]
    named = [p for p in named if not p.strip().startswith(("*args", "**kwargs", "*,"))]
    typed_kwargs = "**kwargs:" in sig
    if "*args" in sig and not typed_kwargs and len(named) <= 3:
        return f"BARE passthrough (*args, {len(named)} named)"
    return f"{len(named)} named params, " + ("typed **kwargs" if typed_kwargs else "untyped **kwargs")


via_twin = top100[top100["hand_norm"] & ~top100["hand_exact"] & top100["obj"].str.startswith("matplotlib.pyplot.")]
rows = []
for r in via_twin.itertuples():
    short = r.obj.rsplit(".", 1)[1]
    rows.append(
        {
            "rank": r.rank,
            "page": r.obj,
            "credited to": r.matched_via,
            "our pyplot.pyi stub": stub_shape(stub_signature(PYPLOT_PYI, short)),
        }
    )
via_twin_df = pd.DataFrame(rows)
n_bare = via_twin_df["our pyplot.pyi stub"].str.startswith(("BARE", "absent")).sum()
n_typed_kwargs = via_twin_df["our pyplot.pyi stub"].str.contains(r"(?<!un)typed \*\*kwargs").sum()
print(f"{len(via_twin_df)} pyplot pages whose Axes/Figure twin is annotated but whose own stub is not.")
print(f"{n_bare} are bare *args passthroughs, and {n_typed_kwargs} have a typed **kwargs — the one thing")
print("that expands a splat into editable properties — so none renders the panel we built.\n")
print(via_twin_df.to_string(index=False))

21 pyplot pages whose Axes/Figure twin is annotated but whose own stub is not.
4 are bare *args passthroughs, and 0 have a typed **kwargs — the one thing
that expands a splat into editable properties — so none renders the panel we built.

 rank                          page                       credited to               our pyplot.pyi stub
    1        matplotlib.pyplot.plot         matplotlib.axes.Axes.plot BARE passthrough (*args, 3 named)
    2      matplotlib.pyplot.legend       matplotlib.axes.Axes.legend BARE passthrough (*args, 0 named)
    3     matplotlib.pyplot.scatter      matplotlib.axes.Axes.scatter 14 named params, untyped **kwargs
    5        matplotlib.pyplot.hist         matplotlib.axes.Axes.hist 16 named params, untyped **kwargs
    6         matplotlib.pyplot.bar          matplotlib.axes.Axes.bar  6 named params, untyped **kwargs
   12     matplotlib.pyplot.boxplot      matplotlib.axes.Axes.boxplot 28 named params, untyped **kwargs
   14        matplotlib.pyplot.te

### …plus the wrappers matplotlib *renamed*

`plt.title` is `Axes.set_title`, `plt.xlim` is `Axes.set_xlim`, `plt.subplot` is
`Figure.add_subplot`. Same-name normalization misses these, so they need a hand-written alias table.
They are un-annotated wrappers of annotated functions too, and they size the gap the same way.

In [11]:
RENAMED_WRAPPERS = {
    "matplotlib.pyplot.title": "matplotlib.axes.Axes.set_title",
    "matplotlib.pyplot.xlabel": "matplotlib.axes.Axes.set_xlabel",
    "matplotlib.pyplot.ylabel": "matplotlib.axes.Axes.set_ylabel",
    "matplotlib.pyplot.xlim": "matplotlib.axes.Axes.set_xlim",
    "matplotlib.pyplot.ylim": "matplotlib.axes.Axes.set_ylim",
    "matplotlib.pyplot.xticks": "matplotlib.axes.Axes.set_xticks",
    "matplotlib.pyplot.yticks": "matplotlib.axes.Axes.set_yticks",
    "matplotlib.pyplot.subplot": "matplotlib.figure.Figure.add_subplot",
    "matplotlib.pyplot.savefig": "matplotlib.figure.Figure.savefig",  # not annotated; listed for completeness
    "matplotlib.pyplot.xscale": "matplotlib.axes.Axes.set_xscale",  # not annotated
    "matplotlib.pyplot.yscale": "matplotlib.axes.Axes.set_yscale",  # not annotated
}

top100["hand_alias"] = top100["hand_norm"] | top100["obj"].map(
    lambda o: RENAMED_WRAPPERS.get(o, "") in hand_exact
)
n_alias = int(top100["hand_alias"].sum())

gained = top100[top100["hand_alias"] & ~top100["hand_norm"]]
print(f"{len(gained)} further pages wrap a function we annotated under a different name:\n")
print(
    gained.assign(wraps=gained["obj"].map(RENAMED_WRAPPERS))[["rank", "obj", "visitors", "wraps"]].to_string(index=False)
)
print("\nour pyplot.pyi stubs for those, same reality check as above:")
for r in gained.itertuples():
    print(f"  {r.obj:<35} {stub_shape(stub_signature(PYPLOT_PYI, r.obj.rsplit('.', 1)[1]))}")

gap = n_alias - n_exact
gap_traffic = top100.loc[top100["hand_alias"] & ~top100["hand_exact"], "visitors"].sum()
print(
    f"\nGap: {gap} of the top 100 pages ({gap_traffic:,.0f} visitors, "
    f"{gap_traffic / top100['visitors'].sum():.0%} of top-100 traffic) are pyplot wrappers of work"
)
print(f"we already did. Delegating those stubs to their twins would take coverage {n_exact} -> {n_alias}.")

8 further pages wrap a function we annotated under a different name:

 rank                       obj  visitors                                wraps
   16 matplotlib.pyplot.subplot    4100.0 matplotlib.figure.Figure.add_subplot
   18  matplotlib.pyplot.xticks    3600.0      matplotlib.axes.Axes.set_xticks
   39    matplotlib.pyplot.xlim    1900.0        matplotlib.axes.Axes.set_xlim
   43   matplotlib.pyplot.title    1800.0       matplotlib.axes.Axes.set_title
   47    matplotlib.pyplot.ylim    1600.0        matplotlib.axes.Axes.set_ylim
   48  matplotlib.pyplot.xlabel    1600.0      matplotlib.axes.Axes.set_xlabel
   78  matplotlib.pyplot.yticks     855.0      matplotlib.axes.Axes.set_yticks
   95  matplotlib.pyplot.ylabel     667.0      matplotlib.axes.Axes.set_ylabel

our pyplot.pyi stubs for those, same reality check as above:
  matplotlib.pyplot.subplot           BARE passthrough (*args, 0 named)
  matplotlib.pyplot.xticks            2 named params, untyped **kwargs
  matplotlib.p

## 9. X and Y for the paper sentence

In [12]:
X = len(hand)


def traffic_share(col):
    return top100.loc[top100[col], "visitors"].sum() / top100["visitors"].sum()


print(f"X = {X}   functions we hand-annotated")
print(f"Y = {n_exact}   of the top 100 function doc pages — the call each page documents lands on a")
print(f"       stub we hand-annotated, so the user gets the panel. {traffic_share('hand_exact'):.0%} of top-100 traffic.")
print(f"\n{n_exact} of the {X} annotated functions are in the top 100; the other {X - n_exact} are listed in C.")

print("\nlooser counts, for the limitations paragraph only — NOT the headline:")
for label, col, n in [
    ("also crediting same-name pyplot wrappers we did not annotate", "hand_norm", n_norm),
    ("also crediting renamed ones (plt.title for Axes.set_title)", "hand_alias", n_alias),
]:
    print(f"  {n:>3}/100 pages, {traffic_share(col):>4.0%} of traffic   — {label}")

pyplot_public = [a for a in dir(importlib.import_module("matplotlib.pyplot")) if not a.startswith("_")]
axes_public = [a for a in dir(importlib.import_module("matplotlib.axes").Axes) if not a.startswith("_")]
print(f"\nfor the 'vanishingly small portion' clause: matplotlib.pyplot exports {len(pyplot_public)}")
print(f"public names and Axes has {len(axes_public)} public methods, before the rest of the API.")

X = 47   functions we hand-annotated
Y = 30   of the top 100 function doc pages — the call each page documents lands on a
       stub we hand-annotated, so the user gets the panel. 28% of top-100 traffic.

30 of the 47 annotated functions are in the top 100; the other 17 are listed in C.

looser counts, for the limitations paragraph only — NOT the headline:
   51/100 pages,  69% of traffic   — also crediting same-name pyplot wrappers we did not annotate
   59/100 pages,  74% of traffic   — also crediting renamed ones (plt.title for Axes.set_title)

for the 'vanishingly small portion' clause: matplotlib.pyplot exports 247
public names and Axes has 298 public methods, before the rest of the API.


## 10. The same question for the top 50

The cutoff at 100 is arbitrary, so here is the whole thing again at 50, plus 25 and "all of them" to
show which way the number moves.

It does not move monotonically: **28% at 25, 36% at 50, 30% at 100.** The reason is visible in the
lists below. Ranks 1–12 are almost entirely `pyplot` pages (`plt.plot`, `plt.legend`, `plt.scatter`,
`plt.hist`, `plt.bar`), which the strict rule excludes; ranks 13–50 are where the `Axes` pages
cluster (`Axes.plot`, `Axes.legend`, `Axes.scatter`, `Axes.text`, `Axes.set_xticks`, …), nearly all of
which we annotated; past 50 the tail thins out into functions we never touched. So the peak at 50 is
a fact about how matplotlib's docs are read, not about our aim.

Which means 50 is the most flattering cutoff available, and a reader who checks the neighbouring ones
will see that. Quote 100, or quote both.

The cutoff sweep does turn up a better statistic, though. Split the pages by calling surface and the
picture sharpens a lot: in ranks 13–50, we annotated **15 of the 16** pages documenting an
`Axes`/`Figure` method and **0 of the 22** `pyplot` pages. Restricted to the object-oriented API that
Plottery is built around, coverage is 94% of the top 50 and 67% of the top 100. That is a claim about
our aim rather than about matplotlib's `pyplot` shim, and it is the one the data actually supports.

In [13]:
def coverage_at(n: int) -> dict:
    """Re-run the whole match against the top `n` function pages."""
    top = merged.head(n).copy()
    top["hand_exact"] = top["obj"].isin(hand_exact)
    top["hand_norm"] = top["obj"].map(normalize).isin(hand_norm)
    top["hand_alias"] = top["hand_norm"] | top["obj"].map(
        lambda o: RENAMED_WRAPPERS.get(o, "") in hand_exact
    )
    total = top["visitors"].sum()
    out = {"n": len(top)}
    for col in ("hand_exact", "hand_norm", "hand_alias"):
        out[col] = int(top[col].sum())
        out[col + "_pct"] = 100 * top[col].sum() / len(top)
        out[col + "_traffic"] = 100 * top.loc[top[col], "visitors"].sum() / total
    return out


cutoffs = pd.DataFrame([coverage_at(n) for n in (25, 50, 100, len(merged))])
print("Y at several cutoffs (headline rule = hand_exact):\n")
print(
    cutoffs.assign(
        Y=lambda d: d["hand_exact"].astype(str) + "/" + d["n"].astype(str),
        Y_pct=lambda d: d["hand_exact_pct"].round(0).astype(int).astype(str) + "%",
        Y_traffic=lambda d: d["hand_exact_traffic"].round(0).astype(int).astype(str) + "%",
        loose=lambda d: d["hand_norm"].astype(str) + "/" + d["n"].astype(str),
        loosest=lambda d: d["hand_alias"].astype(str) + "/" + d["n"].astype(str),
    )[["n", "Y", "Y_pct", "Y_traffic", "loose", "loosest"]].to_string(index=False)
)
print("\nY_pct is the paper's number; Y_traffic is the same pages' share of that cutoff's visitors.")
print("loose/loosest credit un-annotated pyplot wrappers of annotated twins (sections 8-9).")

Y at several cutoffs (headline rule = hand_exact):

  n      Y Y_pct Y_traffic  loose loosest
 25   7/25   28%       24%  17/25   19/25
 50  18/50   36%       29%  30/50   36/50
100 30/100   30%       28% 51/100  59/100
104 30/104   29%       28% 51/104  59/104

Y_pct is the paper's number; Y_traffic is the same pages' share of that cutoff's visitors.
loose/loosest credit un-annotated pyplot wrappers of annotated twins (sections 8-9).


In [14]:
top50 = top100.head(50)
c50 = coverage_at(50)

show(
    "A50. hand-annotated, in the top 50",
    [f"{r.rank:>4}  {r.obj:<46} {r.visitors:>9,.0f}" for r in top50[top50["hand_exact"]].itertuples()],
)
show(
    "B50. not hand-annotated, in the top 50",
    [
        f"{r.rank:>4}  {r.obj:<46} {r.visitors:>9,.0f}"
        + (f"   <- wrapper of {r.matched_via}" if r.hand_norm else "")
        + (f"   <- wrapper of {RENAMED_WRAPPERS[r.obj]}" if not r.hand_norm and r.hand_alias else "")
        for r in top50[~top50["hand_exact"]].itertuples()
    ],
)

only_51_100 = top100.iloc[50:]
print(f"\ntop 50: {int(top50['hand_exact'].sum())}/50 covered ({c50['hand_exact_pct']:.0f}%)")
print(f"ranks 51-100: {int(only_51_100['hand_exact'].sum())}/50 covered, hence 30/100 overall")

print("\nwhere the two calling surfaces sit in the ranking:")
surface = lambda o: "pyplot" if o.startswith("matplotlib.pyplot.") else "Axes/Figure/other"
band = pd.cut(top100["rank"], [0, 12, 50, 100], labels=["1-12", "13-50", "51-100"])
print(
    top100.assign(surface=top100["obj"].map(surface), band=band)
    .pivot_table(index="band", columns="surface", values="hand_exact", aggfunc=["size", "sum"], observed=True)
    .to_string()
)
print("(`size` = pages in that band, `sum` = of those, how many we hand-annotated)")

print(
    f"\nwrapper gap in the top 50: {c50['hand_alias'] - c50['hand_exact']} pages, "
    f"{c50['hand_alias_traffic'] - c50['hand_exact_traffic']:.0f}% of top-50 traffic"
)
print("— proportionally worse than the top 100's 29 pages / 46%, since the head of the list is pyplot.")

# The denominator Plottery actually aims at: pages documenting an Axes or Figure method.
oo = top100[top100["obj"].str.match(r"matplotlib\.(axes\.Axes|figure\.Figure)\.")]
print("\nrestricted to the object-oriented API we target (Axes/Figure methods only):")
for n in (50, 100):
    sub = oo[oo["rank"] <= n]
    print(
        f"  top {n:>3}: {int(sub['hand_exact'].sum())}/{len(sub)} "
        f"({100 * sub['hand_exact'].sum() / len(sub):.0f}%) of the Axes/Figure pages are hand-annotated"
    )
missed = oo[(oo["rank"] <= 100) & ~oo["hand_exact"]]
print(f"\nthe {len(missed)} Axes/Figure pages in the top 100 we did not annotate:")
for r in missed.itertuples():
    print(f"  {r.rank:>4}  {r.obj:<46} {r.visitors:>9,.0f}")


A50. hand-annotated, in the top 50  (18)
   4  matplotlib.pyplot.subplots                        14,000
   9  matplotlib.pyplot.colorbar                         7,600
  10  matplotlib.pyplot.figure                           7,400
  13  matplotlib.axes.Axes.plot                          5,500
  17  matplotlib.axes.Axes.legend                        3,900
  20  matplotlib.axes.Axes.scatter                       3,400
  22  matplotlib.axes.Axes.text                          3,400
  26  matplotlib.axes.Axes.set_xticks                    2,800
  29  matplotlib.axes.Axes.tick_params                   2,500
  31  matplotlib.axes.Axes.set_xlim                      2,300
  32  matplotlib.figure.Figure.add_subplot               2,200
  34  matplotlib.axes.Axes.set_title                     2,100
  36  matplotlib.axes.Axes.annotate                      2,000
  38  matplotlib.axes.Axes.bar                           1,900
  40  matplotlib.axes.Axes.set_ylim                      1,800
  41  matplot

## 11. Things to know before quoting these numbers

**Y = 30, and the reason it is not 51.** The `pyplot` wrapper and the `Axes` method document the
same operation, so it is tempting to credit `Axes.plot` for the `plt.plot` page. But Plottery reads
whichever stub mypy resolves for the call the user typed (§6), and our `pyplot.pyi` wrappers were
never annotated: `plt.plot` and `plt.legend` — the two most-visited function pages on the site — are
still bare `*args, **kwargs`, and not one of the 21 twin-annotated wrappers has a typed `**kwargs`
(§8). Checked in the running UI: `plt.plot([1,2,3],[2,1,3])` gets 4 property rows and misbinds the
y-data to `scalex`, while `ax.plot(...)` gets 35. A user who writes `plt.plot(...)` does not get the
panel we built, so those pages do not count.

**Y is sensitive to the cutoff, and not monotonically** (§10): 28% at 25, 36% at 50, 30% at 100, 29%
over all 104 function pages in the sample. 50 is the best-looking cutoff and 100 is the roundest. Do
not present 36% without noting that 100 gives 30%, because the first reviewer to move the cutoff will
find it.

**The stronger claim available from this data** is per calling surface, not per cutoff: of the top-50
pages documenting an `Axes`/`Figure` method, we hand-annotated **15 of 16** (94%); over the top 100,
26 of 39 (67%). The 13 we missed in the top 100 are listed in §10 — `Axes.imshow` is the only one in
the top 50, and `Axes.fill_between`, `Axes.errorbar`, `Axes.pcolormesh`, and `Axes.contour` are the
notable others. This framing says something about our aim; the raw 30% mostly says that matplotlib's
readers use `pyplot`.

**The gap is worth stating, because it is fixable.** 29 of the top 100 pages, 46% of top-100
traffic, are `pyplot` wrappers of functions we already annotated. Delegating those stubs to their
`Axes`/`Figure` twins would move Y from 30 to 59 without any new annotation work. That is either a
limitation to admit or a to-do before submission — and the `scalex`-misbinding above is a bug on the
most-visited page in matplotlib's API docs, so it is worth fixing on its own merits.

**List B is not "renders nothing".** Excluded pages still get whatever upstream's annotations give —
`pyplot.scatter` has 14 named parameters from upstream, for instance. What they do not get is *our*
work: a `**kwargs` TypedDict expanded into editable properties, real defaults, `Literal` dropdowns.
Y measures our contribution, not absolute GUI quality.

**"Pages" and "functions" are not interchangeable.** X counts functions we annotated; Y counts doc
pages. They coincide here (30 of the 47 annotated functions have a top-100 page, and each covered
page is a distinct function) only because the strict rule stops one function from claiming two pages.

**11 of the 47 are tagged `suspect`** in the annotated diff — real hand-authoring with apparent bugs
(inverted `boxplot` defaults, `loglog(base=1.0)`, a type expression used as a default in
`secondary_xaxis`). Those functions do render a panel, which is what Y measures, but do not describe
them as *correct*. See `README.md`.

**`Axes.hist` is annotated twice** (two overloads), which is why 115 `### ANN` lines yield 114
distinct names.

**List C, the 17 annotated functions outside the top 100, splits in two.** Seven are annotated
`Axes` methods whose only popular page is the `pyplot` twin (`axis`, `axline`, `axvspan`, `barh`,
`hlines`, `loglog`, `Figure.suptitle`) — the same wrapper gap seen from the other side. The other
ten have no doc page in the top 400 under any spelling: `axhspan`, `eventplot`, `indicate_inset`,
`plot_date` (deprecated), `secondary_xaxis`, `secondary_yaxis`, `semilogx`, `semilogy`,
`set_yticklabels`, `xcorr`.

**The denominator drifts.** The CSV is one 28-day window scraped on 2026-07-24, and its top 400
covers only ~104 distinct function pages, so "top 100 function pages" is close to "every function
page anyone visits much". Traffic is also version-split (`pyplot.plot` had three URLs, merged here)
and the traffic counts are rounded to three significant figures in the source table, so treat the
traffic shares as approximate.

**Excluded from the denominator on purpose:** 16 class pages (`Line2D`, `Colormap`, `Rectangle`, …)
and the `pyplot` module page. If a reviewer thinks a class page counts as documenting a callable
(the constructor), the denominator grows and Y falls. Our stubs' `Props` TypedDicts do a lot of work
for exactly those classes, so there is an argument either way — but the sentence says *function*
pages, so they are out.